In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, desc, broadcast
import time

spark = SparkSession.builder\
    .master("spark://192.168.2.156:7077") \
    .appName("Jakob_Ekholm_Project")\
    .config("spark.dynamicAllocation.enabled", True)\
    .config("spark.dynamicAllocation.maxExecutors", 3)\
    .config("spark.executor.instances", 3)\
    .config("spark.executor.cores", 4)\
    .config("spark.executor.memory", "4g")\
    .config("spark.driver.cores", 4)\
    .config("spark.driver.memory", "4g")\
    .config("spark.cores.max", 12)\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [5]:

start_time = time.time()

# Read data
reddit = spark.read.json("hdfs://192.168.2.156:9000/data/reddit/reddit_100k.json")
name_df = spark.read.csv("hdfs://192.168.2.156:9000/output/group31/Formatted_Celebrities.csv", 
                         header=True, inferSchema=True).select(col("name").alias("name"))

reddit_selected = reddit.select("normalizedBody", "subreddit", "summary").dropna()
reddit_selected = reddit_selected.withColumn("normalizedBody", lower(col("normalizedBody")))
reddit_selected = reddit_selected.repartition(16)  # Distribute workload

# Use Broadcast Join Instead of Cross Join
joined_df = reddit_selected.join(broadcast(name_df), reddit_selected["normalizedBody"].contains(name_df["name"]))

# Count occurrences
count_df = joined_df.groupBy("name").count().withColumnRenamed("count", "times")
sorted_df = count_df.orderBy(desc("times"))

# Save results
output_path = "hdfs://192.168.2.156:9000/output/group31/celebrity_counts_sorted.csv"
sorted_df.write.csv(output_path, header=True, mode="overwrite")

end_time = time.time()
print(f"Execution Time: {end_time - start_time:.2f} seconds")
spark.stop()

Execution Time: 31.77 seconds
